In [6]:
import numpy as np
import pandas as pd
import string
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
train_data = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv') 

In [3]:
print(f"Columns of the train dataset : {train_data.columns}")

Columns of the train dataset : Index(['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'], dtype='object')


1. Calculate the frequency distribution of the correct answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent opt

In [4]:
answer_counts = train_data['answer'].value_counts()
print("Frequency Distribution of the correct answers : \n", answer_counts)

result = answer_counts.max() + answer_counts.min()
print("The sum of the occurrences of the most frequent \
option and the least frequent option:", result)

Frequency Distribution of the correct answers : 
 answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
The sum of the occurrences of the most frequent option and the least frequent option: 814


2. After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?

In [7]:
translator = str.maketrans('', '', string.punctuation)

clean_prompts = (
    train_data['prompt']
    .fillna('')
    .str.lower()
    .str.translate(translator)
)

vocab = set()

for text in clean_prompts:
    vocab.update(text.split())

print(f"Total number of unique words (vocabulary size) across \
the entire cleaned prompt column : {len(vocab)}")

Total number of unique words (vocabulary size) across the entire cleaned prompt column : 859


3. Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?

In [9]:
display(train_data[train_data['id']==1])
stop_words  = set(ENGLISH_STOP_WORDS)
words = clean_prompts[0].split()

filtered_words = [word for word in words if word not in stop_words]
print(f"Before filtering words : {words}")
print(f"Filtered Words : {filtered_words}")
print(f"Words are left in the prompt for Row ID 1 after filtering : {len(filtered_words)}")

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B


Before filtering words : ['pick', 'the', 'best', 'possible', 'answer', 'what', 'is', 'martin', 'heideggers', 'view', 'on', 'the', 'relationship', 'between', 'time', 'and', 'human', 'existence', 'among', 'the', 'listed', 'options']
Filtered Words : ['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']
Words are left in the prompt for Row ID 1 after filtering : 13


4. 4. Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?

In [10]:
combined_text = (
    train_data['prompt'].fillna('') + ' ' +
    train_data['A'].fillna('') + ' ' +
    train_data['B'].fillna('') + ' ' +
    train_data['C'].fillna('') + ' ' +
    train_data['D'].fillna('') + ' ' +
    train_data['E'].fillna('')
)

vectorizer = TfidfVectorizer(stop_words='english')

X = vectorizer.fit_transform(combined_text)

vocab_size = len(vectorizer.vocabulary_)

print("Vocabulary size:", vocab_size)

Vocabulary size: 2762


5. Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).

In [11]:
# Row with id = 1
row = train_data[train_data['id'] == 1].iloc[0]

prompt = row['prompt']
option_a = row['A']

# Used the vectorizer already fitted in Q4
prompt_vec = vectorizer.transform([prompt])
option_a_vec = vectorizer.transform([option_a])

similarity = cosine_similarity(prompt_vec, option_a_vec)[0][0]

print(round(similarity, 4))

0.272


6. Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.

In [12]:
predictions = []

for _, row in train_data.iterrows():

    texts = [row['prompt'], row['A'], row['B'], row['C'], row['D'], row['E']]
    vectors = vectorizer.transform(texts)
    prompt_vec = vectors[0]
    option_vecs = vectors[1:]

    sims = cosine_similarity(prompt_vec, option_vecs)[0]

    predicted = ['A', 'B', 'C', 'D', 'E'][np.argmax(sims)]
    predictions.append(predicted)
   

accuracy = (train_data['answer'] == predictions).mean() * 100

print(f"Accuracy = {accuracy:.2f}%")

Accuracy = 13.55%


7. If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?  

In [13]:
def find_MAP(truth,pred):
    score = 0
    if truth == pred[0]:
        score = 1.0
    elif truth == pred[1]:
        score = 1/2
    elif truth == pred[2]:
        score = 1/3
    return score

In [14]:
truth = 'C'
pred = ['C','A','B']
print(f"MAP@3 is : {find_MAP(truth,pred)}")

MAP@3 is : 1.0


8. If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E? 

In [15]:
def find_MAP(truth,pred):
    score = 0
    if truth == pred[0]:
        score = 1.0
    elif truth == pred[1]:
        score = 1/2
    elif truth == pred[2]:
        score = 1/3
    return score

In [16]:
truth = 'B'
pred = ['D','B','E']
print(f"MAP@3 is : {find_MAP(truth,pred)}")

MAP@3 is : 0.5


9. The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [17]:
answer_counts = train_data['answer'].value_counts()

print("Answer Frequency Distribution : ")
print(answer_counts)

top3 = answer_counts.index[:3].tolist()

print("Top 3 Answers : ")
print(top3)

def apk(actual, predicted, k=3):
    predicted = predicted[:k]

    if actual in predicted:
        return 1.0 / (predicted.index(actual) + 1)

    return 0.0

map3 = train_data['answer'].apply(
    lambda x: apk(x, top3)
).mean()

print(f"Majority Class Baseline MAP@3 = {map3:.3f}")

Answer Frequency Distribution : 
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Top 3 Answers : 
['B', 'C', 'A']
Majority Class Baseline MAP@3 = 0.421


10. The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?

In [18]:
def apk(actual, predicted, k=3):
    predicted = predicted[:k]

    if actual in predicted:
        return 1.0 / (predicted.index(actual) + 1)

    return 0.0

scores = []

for _, row in train_data.iterrows():

    prompt_vec = vectorizer.transform([row['prompt']])

    similarities = {}

    for option in ['A', 'B', 'C', 'D', 'E']:
        option_vec = vectorizer.transform([row[option]])
        similarities[option] = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

    ranked_options = sorted(
        similarities,
        key=similarities.get,
        reverse=True
    )

    top3 = ranked_options[:3]

    scores.append(
        apk(row['answer'], top3)
    )

map3 = np.mean(scores)

print(f"MAP@3 = {map3:.3f}")

MAP@3 = 0.296
